CLV = Average Order Value * Purchases per Year * Years * Profit Margin

In [1]:
import pandas as pd
MARGIN = 0.30 #you keep 30 paise of profit on every rupee of sales
YEARS = 3 #how far ahead you want to project

In [2]:
df=pd.read_excel(r"C:\Users\nihar\Downloads\RFM for Python.xlsx")
df

,Transaction,Customer,Date,Amount
0,1,4184,2016-09-03,30
1,2,3657,2018-10-05,34
2,3,1011,2016-09-18,47
3,4,106,2015-06-08,94
4,5,739,2017-06-12,73
...,...,...,...,...
99995,99996,3511,2017-10-14,76
99996,99997,3939,2016-06-25,99
99997,99998,3262,2018-08-25,77
99998,99999,668,2018-09-22,92


In [3]:
df.columns = ["Transaction", "Customer", "Date", "Amount"]
df["Date"] = pd.to_datetime(df["Date"])

print("Rows      :", len(df))
print("Customers :", df.Customer.nunique())
print("Period    :", df.Date.min().date(), "to", df.Date.max().date())

Rows      : 100000
Customers : 5000
Period    : 2014-12-06 to 2018-10-06


In [4]:
# 2. SUMMARISE EACH CUSTOMER
today = df.Date.max()

clv = df.groupby("Customer").agg(
    Orders=("Amount", "count"),        # how many times they bought
    TotalSpend=("Amount", "sum"),      # how much they spent in total
    FirstBuy=("Date", "min"),
    LastBuy=("Date", "max"),
)

clv

,Orders,TotalSpend,FirstBuy,LastBuy
Customer,,,,
1,15,937,2015-01-05,2018-08-06
2,20,1390,2014-12-17,2018-09-01
3,19,1337,2015-02-03,2018-09-04
4,22,1549,2015-03-02,2018-09-25
5,13,792,2015-01-17,2018-03-17
...,...,...,...,...
4996,17,1242,2015-02-12,2018-07-26
4997,21,1243,2015-02-10,2018-05-30
4998,17,986,2014-12-30,2017-08-22


In [5]:
# Average Order Value
clv["AOV"] = clv.TotalSpend / clv.Orders

# How long they have been a customer, in years (minimum 1 year to avoid divide-by-zero)
clv["Years_Active"] = ((today - clv.FirstBuy).dt.days / 365).clip(lower=1)  #clip is a function used to round-off.

# Purchases per year
clv["Orders_per_Year"] = clv.Orders / clv.Years_Active

# Days since last purchase
clv["Recency_Days"] = (today - clv.LastBuy).dt.days

clv

,Orders,TotalSpend,FirstBuy,LastBuy,AOV,Years_Active,Orders_per_Year,Recency_Days
Customer,,,,,,,,
1,15,937,2015-01-05,2018-08-06,62.466667,3.753425,3.996350,61
2,20,1390,2014-12-17,2018-09-01,69.500000,3.805479,5.255580,35
3,19,1337,2015-02-03,2018-09-04,70.368421,3.673973,5.171514,32
4,22,1549,2015-03-02,2018-09-25,70.409091,3.600000,6.111111,11
5,13,792,2015-01-17,2018-03-17,60.923077,3.720548,3.494109,203
...,...,...,...,...,...,...,...,...
4996,17,1242,2015-02-12,2018-07-26,73.058824,3.649315,4.658408,72
4997,21,1243,2015-02-10,2018-05-30,59.190476,3.654795,5.745877,129
4998,17,986,2014-12-30,2017-08-22,58.000000,3.769863,4.509448,410


In [6]:
# Parameters
MARGIN = 0.30
YEARS = 3

# 3. THE CLV FORMULA
clv["CLV"] = clv.AOV * clv.Orders_per_Year * YEARS * MARGIN
clv

,Orders,TotalSpend,FirstBuy,LastBuy,AOV,Years_Active,Orders_per_Year,Recency_Days,CLV
Customer,,,,,,,,,
1,15,937,2015-01-05,2018-08-06,62.466667,3.753425,3.996350,61,224.674818
2,20,1390,2014-12-17,2018-09-01,69.500000,3.805479,5.255580,35,328.736501
3,19,1337,2015-02-03,2018-09-04,70.368421,3.673973,5.171514,32,327.520134
4,22,1549,2015-03-02,2018-09-25,70.409091,3.600000,6.111111,11,387.250000
5,13,792,2015-01-17,2018-03-17,60.923077,3.720548,3.494109,203,191.584683
...,...,...,...,...,...,...,...,...,...
4996,17,1242,2015-02-12,2018-07-26,73.058824,3.649315,4.658408,72,306.304054
4997,21,1243,2015-02-10,2018-05-30,59.190476,3.654795,5.745877,129,306.091079
4998,17,986,2014-12-30,2017-08-22,58.000000,3.769863,4.509448,410,235.393169


In [7]:
# 4. GROUP INTO TIERS
clv["Tier"] = pd.qcut(clv.CLV, 4, labels=["Low", "Medium", "High", "Top"])
clv

,Orders,TotalSpend,FirstBuy,LastBuy,AOV,Years_Active,Orders_per_Year,Recency_Days,CLV,Tier
Customer,,,,,,,,,,
1,15,937,2015-01-05,2018-08-06,62.466667,3.753425,3.996350,61,224.674818,Low
2,20,1390,2014-12-17,2018-09-01,69.500000,3.805479,5.255580,35,328.736501,High
3,19,1337,2015-02-03,2018-09-04,70.368421,3.673973,5.171514,32,327.520134,High
4,22,1549,2015-03-02,2018-09-25,70.409091,3.600000,6.111111,11,387.250000,Top
5,13,792,2015-01-17,2018-03-17,60.923077,3.720548,3.494109,203,191.584683,Low
...,...,...,...,...,...,...,...,...,...,...
4996,17,1242,2015-02-12,2018-07-26,73.058824,3.649315,4.658408,72,306.304054,Medium
4997,21,1243,2015-02-10,2018-05-30,59.190476,3.654795,5.745877,129,306.091079,Medium
4998,17,986,2014-12-30,2017-08-22,58.000000,3.769863,4.509448,410,235.393169,Low


In [8]:
# 5. RESULTS
print(f"\n— {YEARS}-Year CLV (margin {MARGIN:.0%}) —")
print("Average CLV :", round(clv.CLV.mean(), 2))
print("Lowest CLV  :", round(clv.CLV.min(), 2))
print("Highest CLV :", round(clv.CLV.max(), 2))
print("Total value :", round(clv.CLV.sum(), 2))

print("\n— By tier —")
print(
    clv.groupby("Tier", observed=True)
    .agg(
        Customers=("CLV", "size"),
        Avg_CLV=("CLV", "mean"),
        Avg_AOV=("AOV", "mean"),
        Avg_Orders_per_Year=("Orders_per_Year", "mean"),
    )
    .round(2)
)

print("\n— Top 10 customers ——")
print(
    clv.sort_values("CLV", ascending=False)[
        ["Orders", "AOV", "Orders_per_Year", "CLV", "Tier"]
    ]
    .head(10)
    .round(2)
)


— 3-Year CLV (margin 30%) —
Average CLV : 321.22
Lowest CLV  : 109.5
Highest CLV : 646.61
Total value : 1606077.14

— By tier —
        Customers  Avg_CLV  Avg_AOV  Avg_Orders_per_Year
Tier                                                    
Low          1250   230.71    62.99                 4.09
Medium       1250   294.28    64.66                 5.08
High         1250   342.29    65.75                 5.81
Top          1250   417.59    66.86                 6.96

— Top 10 customers ——
          Orders    AOV  Orders_per_Year     CLV Tier
Customer                                             
3313          39  70.21            10.23  646.61  Top
3209          38  64.66             9.99  581.08  Top
430           30  70.77             9.09  579.24  Top
4641          33  74.15             8.63  576.23  Top
4725          30  63.97             9.91  570.49  Top
3549          33  71.58             8.76  564.30  Top
1205          33  65.61             9.51  561.33  Top
3998          35  66

In [9]:
# 6. SAVE
clv.drop(columns=["FirstBuy", "LastBuy"]).round(2).to_csv("sample_clv.csv")
print("\nSaved to sample_clv.csv")


Saved to sample_clv.csv
